# Comparación experimental de arquitecturas para riesgo de Glaucoma

Este notebook compara los mejores checkpoints obtenidos mediante
Fine-Tuning para las dos arquitecturas candidatas del modelo
especializado en evaluación de riesgo de Glaucoma:

- ResNet50V2
- EfficientNetV2-M

La comparación utiliza exclusivamente el conjunto Validation.

El conjunto Test permanece completamente aislado y no se utilizará
hasta haber seleccionado la arquitectura y el umbral operativo final.

Etapas:

1. Definición explícita de los criterios de selección.
2. Carga y verificación de rutas de logs de Fine-Tuning.
3. Recuperación del mejor checkpoint por arquitectura según val_loss.
4. Construcción de la tabla comparativa experimental.
5. Cuantificación de diferencias relativas y absolutas.
6. Preparación del conjunto Validation para comparación directa.
7. Inferencia del mejor checkpoint de ResNet50V2 sobre Validation.
8. Inferencia del mejor checkpoint de EfficientNetV2-M sobre Validation.
9. Consolidación de predicciones de Validation.
10. Intervalos de confianza mediante bootstrap pareado.
11. Selección formal de la arquitectura (ResNet50V2).
12. Selección del umbral operativo en Validation.
13. Congelamiento del modelo y umbral operativo.
14. Evaluación final sobre Test.
15. Inferencia y métricas independientes del umbral.
16. Binarización con umbral congelado y métricas finales.
17. Persistencia de resultados de Test.
18. Intervalos de confianza de la evaluación final mediante bootstrap.
19. Conclusión técnica y síntesis final del experimento.


## 2. Criterio de selección de arquitectura

La comparación experimental se realizará utilizando:

1. **ROC-AUC de Validation como criterio principal**, debido a que
   evalúa la capacidad discriminativa del modelo independientemente
   de un umbral fijo.

2. **PR-AUC de Validation como criterio secundario**, debido al
   desbalance existente entre las clases LOW_RISK y HIGH_RISK.

3. **Complejidad computacional como criterio complementario**,
   considerando principalmente el número total de parámetros.

Las métricas Accuracy, Precision y Recall calculadas con umbral 0.5
se presentan únicamente como información descriptiva, ya que el
umbral operativo definitivo será seleccionado posteriormente
utilizando únicamente Validation.

La selección del mejor checkpoint dentro de cada arquitectura se
mantiene basada en la mínima `val_loss`, conforme al protocolo de
entrenamiento previamente establecido.


## 3. Carga de dependencias y rutas de resultados

Definición de las rutas del proyecto hacia los directorios de resultados de ambas arquitecturas y validación de existencia de los archivos de registro (`CSVLogger`) generados durante el Fine-Tuning.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/mnt/c/Users/SeuNg720/Documents/TP1-RetiScan"
)
RESNET_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "glaucoma"
    / "resnet50v2"
)
EFFICIENTNET_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "glaucoma"
    / "efficientnetv2m"
)
RESNET_LOG_PATH = (
    RESNET_RESULTS_DIR
    / "resnet50v2_finetune_log.csv"
)
EFFICIENTNET_LOG_PATH = (
    EFFICIENTNET_RESULTS_DIR
    / "efficientnetv2m_finetune_log.csv"
)

print(
    "ResNet log existe:",
    RESNET_LOG_PATH.exists()
)
print(
    "EfficientNet log existe:",
    EFFICIENTNET_LOG_PATH.exists()
)


ResNet log existe: True
EfficientNet log existe: True


## 4. Recuperación de los mejores checkpoints

Carga de los historiales de Fine-Tuning de cada arquitectura e identificación automática del mejor checkpoint mediante la minimización de `val_loss` en el conjunto Validation.


In [2]:
resnet_history = pd.read_csv(
    RESNET_LOG_PATH
)
efficientnet_history = pd.read_csv(
    EFFICIENTNET_LOG_PATH
)

resnet_best_idx = (
    resnet_history["val_loss"].idxmin()
)
efficientnet_best_idx = (
    efficientnet_history["val_loss"].idxmin()
)

resnet_best = resnet_history.loc[
    resnet_best_idx
]
efficientnet_best = efficientnet_history.loc[
    efficientnet_best_idx
]

resnet_best_epoch = (
    int(resnet_best["epoch"]) + 1
)
efficientnet_best_epoch = (
    int(efficientnet_best["epoch"]) + 1
)

print(
    "Mejor época ResNet50V2:",
    resnet_best_epoch
)
print(
    "Mejor época EfficientNetV2-M:",
    efficientnet_best_epoch
)


Mejor época ResNet50V2: 4
Mejor época EfficientNetV2-M: 6


## 5. Tabla comparativa experimental

Consolidación en un DataFrame estructurado de las métricas clave de validación (`val_loss`, `val_accuracy`, `val_roc_auc`, `val_pr_auc`, `val_precision`, `val_recall`) y del recuento de parámetros totales y entrenables durante el Fine-Tuning.


In [3]:
comparison_df = pd.DataFrame(
    {
        "ResNet50V2": {
            "best_epoch": resnet_best_epoch,
            "val_loss": (
                resnet_best["val_loss"]
            ),
            "val_accuracy": (
                resnet_best["val_accuracy"]
            ),
            "val_roc_auc": (
                resnet_best["val_roc_auc"]
            ),
            "val_pr_auc": (
                resnet_best["val_pr_auc"]
            ),
            "val_precision_0.5": (
                resnet_best["val_precision"]
            ),
            "val_recall_0.5": (
                resnet_best["val_recall"]
            ),
            "total_params": 23_566_849,
            "trainable_params_ft": 14_952_449
        },
        "EfficientNetV2-M": {
            "best_epoch": efficientnet_best_epoch,
            "val_loss": (
                efficientnet_best["val_loss"]
            ),
            "val_accuracy": (
                efficientnet_best["val_accuracy"]
            ),
            "val_roc_auc": (
                efficientnet_best["val_roc_auc"]
            ),
            "val_pr_auc": (
                efficientnet_best["val_pr_auc"]
            ),
            "val_precision_0.5": (
                efficientnet_best["val_precision"]
            ),
            "val_recall_0.5": (
                efficientnet_best["val_recall"]
            ),
            "total_params": 53_151_669,
            "trainable_params_ft": 17_637_261
        }
    }
)

display(
    comparison_df.round(6)
)


,ResNet50V2,EfficientNetV2-M
best_epoch,4.000000e+00,6.000000e+00
val_loss,2.747420e-01,3.560800e-01
val_accuracy,9.009900e-01,8.366340e-01
val_roc_auc,8.878310e-01,8.480420e-01
val_pr_auc,5.261920e-01,4.256100e-01
val_precision_0.5,6.666670e-01,4.210530e-01
val_recall_0.5,5.185190e-01,5.925930e-01
total_params,2.356685e+07,5.315167e+07
trainable_params_ft,1.495245e+07,1.763726e+07


## 6. Diferencias entre arquitecturas

Cálculo de la diferencia aritmética (ResNet50V2 − EfficientNetV2-M) en las métricas de validación para determinar qué modelo exhibe mejor desempeño en los criterios definidos (ROC-AUC y PR-AUC).


In [4]:
metric_differences = pd.DataFrame(
    {
        "Diferencia ResNet - EfficientNet": {
            "val_loss": (
                resnet_best["val_loss"]
                - efficientnet_best["val_loss"]
            ),
            "val_accuracy": (
                resnet_best["val_accuracy"]
                - efficientnet_best["val_accuracy"]
            ),
            "val_roc_auc": (
                resnet_best["val_roc_auc"]
                - efficientnet_best["val_roc_auc"]
            ),
            "val_pr_auc": (
                resnet_best["val_pr_auc"]
                - efficientnet_best["val_pr_auc"]
            ),
            "val_precision_0.5": (
                resnet_best["val_precision"]
                - efficientnet_best["val_precision"]
            ),
            "val_recall_0.5": (
                resnet_best["val_recall"]
                - efficientnet_best["val_recall"]
            )
        }
    }
)

display(
    metric_differences.round(6)
)


,Diferencia ResNet - EfficientNet
val_loss,-0.081337
val_accuracy,0.064356
val_roc_auc,0.039788
val_pr_auc,0.100582
val_precision_0.5,0.245614
val_recall_0.5,-0.074074


## 7. Preparación del conjunto Validation para comparación directa

Se reconstruye el mismo pipeline geométrico utilizado durante el
entrenamiento para procesar las 202 imágenes del conjunto Validation.

No se aplica Data Augmentation durante esta etapa.

Cada arquitectura utilizará únicamente su preprocesamiento específico:

- ResNet50V2: `preprocess_input` de ResNetV2.
- EfficientNetV2-M: imágenes `float32` en `[0, 255]`, debido a que
  `include_preprocessing=True` forma parte del propio modelo.

El orden de las muestras se mantiene fijo (`shuffle=False`) para
garantizar que los scores de ambas arquitecturas correspondan
exactamente a las mismas imágenes.


### 7.1. Carga de dependencias adicionales y partición Validation

Importación de librerías para procesamiento de imágenes y lectura estricta de las 202 muestras congeladas de Validation sin reordenamiento.


In [4]:
from pathlib import Path

import gc
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Ruta raíz del proyecto
PROJECT_ROOT = Path(
    "/mnt/c/Users/SeuNg720/Documents/TP1-RetiScan"
)

CHAKSU_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "glaucoma"
    / "chaksu"
)

SPLITS_PATH = (
    CHAKSU_ROOT
    / "metadata"
    / "chaksu_splits.csv"
)

IMAGE_SIZE = 384
BATCH_SIZE = 8

NUM_PARALLEL_CALLS = 2
PREFETCH_BUFFER = 1

splits_df = pd.read_csv(
    SPLITS_PATH
)

val_df = splits_df[
    splits_df["experimental_split"]
    == "validation"
].copy()

val_df = val_df.reset_index(
    drop=True
)

print(
    "Proyecto:",
    PROJECT_ROOT
)

print(
    "Splits:",
    SPLITS_PATH
)

print(
    "\nImágenes Validation:",
    len(val_df)
)

print(
    "\nDistribución:"
)

print(
    val_df["label"]
    .value_counts()
    .sort_index()
)

Proyecto: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan
Splits: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/glaucoma/chaksu/metadata/chaksu_splits.csv

Imágenes Validation: 202

Distribución:
label
0    175
1     27
Name: count, dtype: int64


### 7.2. Pipeline geométrico común

Reconstrucción idéntica de las etapas de detección de FOV, recorte con margen de 2%, padding cuadrado y redimensionamiento a 384 × 384 con interpolación `INTER_AREA`.


In [5]:
def resolver_ruta(ruta_relativa):
    return PROJECT_ROOT / Path(ruta_relativa)

def detectar_fov(imagen_rgb, umbral=10):
    gris = cv2.cvtColor(
        imagen_rgb,
        cv2.COLOR_RGB2GRAY
    )
    _, mascara_inicial = cv2.threshold(
        gris,
        umbral,
        255,
        cv2.THRESH_BINARY
    )
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (15, 15)
    )
    mascara_limpia = cv2.morphologyEx(
        mascara_inicial,
        cv2.MORPH_CLOSE,
        kernel
    )
    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mascara_limpia,
            connectivity=8
        )
    )
    if num_labels <= 1:
        return None, None
    areas = stats[
        1:,
        cv2.CC_STAT_AREA
    ]
    mayor_componente = (
        1 + np.argmax(areas)
    )
    mascara_componente = (
        labels == mayor_componente
    ).astype(np.uint8) * 255
    contornos, _ = cv2.findContours(
        mascara_componente,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    if len(contornos) == 0:
        return None, None
    contorno_principal = max(
        contornos,
        key=cv2.contourArea
    )
    hull = cv2.convexHull(
        contorno_principal
    )
    mascara_fov = np.zeros_like(
        mascara_componente
    )
    cv2.fillConvexPoly(
        mascara_fov,
        hull,
        255
    )
    x, y, w, h = cv2.boundingRect(
        hull
    )
    return mascara_fov, (x, y, w, h)

def ampliar_bbox(
    bbox,
    image_width,
    image_height,
    margen=0.02
):
    x, y, w, h = bbox
    margen_x = int(w * margen)
    margen_y = int(h * margen)
    x1 = max(
        0,
        x - margen_x
    )
    y1 = max(
        0,
        y - margen_y
    )
    x2 = min(
        image_width,
        x + w + margen_x
    )
    y2 = min(
        image_height,
        y + h + margen_y
    )
    return x1, y1, x2, y2

def recortar_fov(imagen_rgb):
    mascara, bbox = detectar_fov(
        imagen_rgb
    )
    if bbox is None:
        return imagen_rgb, None
    imagen_enmascarada = cv2.bitwise_and(
        imagen_rgb,
        imagen_rgb,
        mask=mascara
    )
    alto, ancho = imagen_rgb.shape[:2]
    x1, y1, x2, y2 = ampliar_bbox(
        bbox,
        image_width=ancho,
        image_height=alto,
        margen=0.02
    )
    recorte = imagen_enmascarada[
        y1:y2,
        x1:x2
    ]
    return recorte, (
        x1,
        y1,
        x2,
        y2
    )

def padding_cuadrado(imagen_rgb):
    alto, ancho = imagen_rgb.shape[:2]
    lado = max(alto, ancho)
    imagen_cuadrada = np.zeros(
        (
            lado,
            lado,
            3
        ),
        dtype=imagen_rgb.dtype
    )
    y_offset = (
        lado - alto
    ) // 2
    x_offset = (
        lado - ancho
    ) // 2
    imagen_cuadrada[
        y_offset:y_offset + alto,
        x_offset:x_offset + ancho
    ] = imagen_rgb
    return imagen_cuadrada

def redimensionar_imagen(
    imagen_rgb,
    size=IMAGE_SIZE
):
    return cv2.resize(
        imagen_rgb,
        (size, size),
        interpolation=cv2.INTER_AREA
    )

def preprocesar_geometria(
    imagen_rgb,
    size=IMAGE_SIZE
):
    recorte, bbox = recortar_fov(
        imagen_rgb
    )
    if bbox is None:
        raise ValueError(
            "No se pudo detectar el campo "
            "de visión retinal."
        )
    imagen_cuadrada = padding_cuadrado(
        recorte
    )
    imagen_resize = redimensionar_imagen(
        imagen_cuadrada,
        size=size
    )
    return imagen_resize, bbox


### 7.3. Funciones de carga y dataset base

Integración con `tf.data` manteniendo el orden exacto del DataFrame (`shuffle=False`) para garantizar que la correspondencia entre muestras sea idéntica.


In [6]:
def cargar_y_preprocesar_python(
    image_path,
    label
):
    ruta = resolver_ruta(
        image_path.decode("utf-8")
    )
    with Image.open(ruta) as img:
        imagen_rgb = np.array(
            img.convert("RGB")
        )
    procesada, _ = preprocesar_geometria(
        imagen_rgb,
        size=IMAGE_SIZE
    )
    procesada = procesada.astype(
        np.float32
    )
    return (
        procesada,
        np.float32(label)
    )

def cargar_y_preprocesar_tf(
    image_path,
    label
):
    imagen, etiqueta = tf.numpy_function(
        func=cargar_y_preprocesar_python,
        inp=[
            image_path,
            label
        ],
        Tout=[
            tf.float32,
            tf.float32
        ]
    )
    imagen.set_shape(
        [IMAGE_SIZE, IMAGE_SIZE, 3]
    )
    etiqueta.set_shape([])
    return imagen, etiqueta

def crear_dataset_base(
    dataframe,
    shuffle=False
):
    rutas = dataframe[
        "image_path"
    ].astype(str).values
    etiquetas = dataframe[
        "label"
    ].astype(np.float32).values
    ds = tf.data.Dataset.from_tensor_slices(
        (
            rutas,
            etiquetas
        )
    )
    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )
    ds = ds.map(
        cargar_y_preprocesar_tf,
        num_parallel_calls=NUM_PARALLEL_CALLS
    )
    return ds


### 7.4. Creación de los datasets de Validation

Instanciación de los pipelines específicos para cada arquitectura: normalización a `[-1, 1]` con `preprocess_resnet50v2` para ResNet50V2, y mantenimiento del rango `[0, 255]` (`float32`) para EfficientNetV2-M.


In [7]:
def preprocess_resnet50v2(
    imagen,
    etiqueta
):
    imagen = (
        tf.keras.applications
        .resnet_v2
        .preprocess_input(
            imagen
        )
    )
    return imagen, etiqueta

def preprocess_efficientnetv2(
    imagen,
    etiqueta
):
    imagen = tf.cast(
        imagen,
        tf.float32
    )
    return imagen, etiqueta

val_ds_resnet = (
    crear_dataset_base(
        val_df,
        shuffle=False
    )
    .map(
        preprocess_resnet50v2,
        num_parallel_calls=NUM_PARALLEL_CALLS
    )
    .batch(BATCH_SIZE)
    .prefetch(PREFETCH_BUFFER)
)

val_ds_efficientnet = (
    crear_dataset_base(
        val_df,
        shuffle=False
    )
    .map(
        preprocess_efficientnetv2,
        num_parallel_calls=NUM_PARALLEL_CALLS
    )
    .batch(BATCH_SIZE)
    .prefetch(PREFETCH_BUFFER)
)


W0000 00:00:1790716529.361018  121046 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1790716529.381804  121046 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1790716529.644342  121046 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9217 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:01:00.0, compute capability: 12.0a


### 7.5. Verificación de tensores de entrada y alineación de etiquetas

Inspección de un lote de prueba para comprobar las dimensiones espaciales `(8, 384, 384, 3)`, los rangos numéricos de píxeles y la perfecta equivalencia de etiquetas entre ambos flujos de datos.


In [8]:
for images_r, labels_r in val_ds_resnet.take(1):
    print(
        "ResNet shape:",
        images_r.shape
    )
    print(
        "ResNet min/max:",
        float(tf.reduce_min(images_r)),
        float(tf.reduce_max(images_r))
    )

for images_e, labels_e in (
    val_ds_efficientnet.take(1)
):
    print(
        "\nEfficientNet shape:",
        images_e.shape
    )
    print(
        "EfficientNet min/max:",
        float(tf.reduce_min(images_e)),
        float(tf.reduce_max(images_e))
    )

print(
    "\nMismas etiquetas primer batch:",
    np.array_equal(
        labels_r.numpy(),
        labels_e.numpy()
    )
)


ResNet shape: (8, 384, 384, 3)
ResNet min/max: -1.0 1.0

EfficientNet shape: (8, 384, 384, 3)
EfficientNet min/max: 0.0 255.0

Mismas etiquetas primer batch: True


## 8. Inferencia del mejor checkpoint de ResNet50V2 sobre Validation

Se carga el mejor checkpoint obtenido durante Fine-Tuning de
ResNet50V2 y se generan scores continuos para las 202 imágenes del
conjunto Validation.

El conjunto Test permanece sin utilizar.

Las predicciones se almacenan en disco antes de liberar el modelo de
la memoria GPU, evitando mantener simultáneamente ambas arquitecturas
cargadas.


### 8.1. Rutas del checkpoint y resultados

Definición de rutas de persistencia para las predicciones en `results/glaucoma/model_comparison` y verificación de disponibilidad del mejor checkpoint de Fine-Tuning de ResNet50V2.


In [9]:
COMPARISON_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "glaucoma"
    / "model_comparison"
)
COMPARISON_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESNET_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "glaucoma"
    / "resnet50v2"
    / "resnet50v2_finetune_best.keras"
)
RESNET_SCORES_PATH = (
    COMPARISON_RESULTS_DIR
    / "resnet50v2_validation_scores.csv"
)

print(
    "Checkpoint ResNet:",
    RESNET_MODEL_PATH
)
print(
    "Existe:",
    RESNET_MODEL_PATH.exists()
)
print(
    "Salida:",
    RESNET_SCORES_PATH
)


Checkpoint ResNet: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/models/glaucoma/resnet50v2/resnet50v2_finetune_best.keras
Existe: True
Salida: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/resnet50v2_validation_scores.csv


### 8.2. Carga del modelo ResNet50V2

Carga del modelo preentrenado y ajustado finamente sin recompilar el optimizador, verificando el nombre de la arquitectura y la cantidad total de parámetros.


In [10]:
resnet_model = tf.keras.models.load_model(
    RESNET_MODEL_PATH,
    compile=False
)

print(
    "Modelo cargado:",
    resnet_model.name
)
print(
    "Parámetros:",
    f"{resnet_model.count_params():,}"
)


Modelo cargado: ResNet50V2_Chaksu
Parámetros: 23,566,849


### 8.3. Generación de scores continuos sobre Validation

Inferencia sobre `val_ds_resnet` para obtener las probabilidades sigmoid de riesgo de glaucoma en las 202 imágenes de validación y comprobación de integridad numérica (ausencia de valores `NaN`).


In [11]:
resnet_scores = resnet_model.predict(
    val_ds_resnet,
    verbose=1
)
resnet_scores = (
    resnet_scores
    .reshape(-1)
)

y_true = (
    val_df["label"]
    .astype(int)
    .to_numpy()
)

print(
    "Scores generados:",
    len(resnet_scores)
)
print(
    "Etiquetas:",
    len(y_true)
)
print(
    "Score mínimo:",
    float(resnet_scores.min())
)
print(
    "Score máximo:",
    float(resnet_scores.max())
)
print(
    "NaN:",
    int(np.isnan(resnet_scores).sum())
)


I0000 00:00:1790717038.583560  121569 service.cc:153] XLA service 0x79692405bda0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790717038.583612  121569 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5070, Compute Capability 12.0a (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.26.0)
I0000 00:00:1790717038.715430  121569 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790717039.873673  121569 cuda_dnn.cc:461] Loaded cuDNN version 92600
I0000 00:00:1790717048.747930  121569 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 802ms/step
Scores generados: 202
Etiquetas: 202
Score mínimo: 0.0002009407471632585
Score máximo: 0.9674186706542969
NaN: 0


### 8.4. Evaluación discriminativa directa (ROC-AUC y Average Precision)

Cálculo independiente mediante `scikit-learn` del ROC-AUC exacto y Average Precision (AP), verificando la coherencia con las métricas registradas en Keras.


In [12]:
resnet_roc_auc_exact = roc_auc_score(
    y_true,
    resnet_scores
)
resnet_average_precision = (
    average_precision_score(
        y_true,
        resnet_scores
    )
)

print(
    "ROC-AUC directo:",
    f"{resnet_roc_auc_exact:.6f}"
)
print(
    "Average Precision:",
    f"{resnet_average_precision:.6f}"
)


ROC-AUC directo: 0.887407
Average Precision: 0.544996


### 8.5. Persistencia de predicciones individuales

Guardado en disco del archivo `resnet50v2_validation_scores.csv` asociando cada nombre de imagen con su etiqueta verdadera y el score continuo emitido por el modelo.


In [13]:
resnet_validation_results = pd.DataFrame(
    {
        "image_name": val_df[
            "image_name"
        ].values,
        "label": y_true,
        "resnet50v2_score": (
            resnet_scores
        )
    }
)
resnet_validation_results.to_csv(
    RESNET_SCORES_PATH,
    index=False
)

print(
    "Archivo guardado en:",
    RESNET_SCORES_PATH
)
display(
    resnet_validation_results.head()
)


Archivo guardado en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/resnet50v2_validation_scores.csv


,image_name,label,resnet50v2_score
0,IMG_2544.JPG,0,0.006071
1,IMG_3380.JPG,0,0.000201
2,IMG_3506.JPG,0,0.021984
3,IMG_3625.JPG,0,0.014642
4,Image188.JPG,1,0.438085


### 8.6. Liberación de memoria GPU

Eliminación de la referencia al modelo, limpieza de la sesión de TensorFlow y ejecución del recolector de basura (`gc.collect`) para vaciar la VRAM antes de instanciar el siguiente modelo.


In [14]:
del resnet_model
tf.keras.backend.clear_session()
gc.collect()

print(
    "ResNet50V2 liberado de memoria."
)


ResNet50V2 liberado de memoria.


## 9. Inferencia del mejor checkpoint de EfficientNetV2-M sobre Validation

Se carga de forma independiente el mejor checkpoint obtenido durante
Fine-Tuning de EfficientNetV2-M y se generan scores continuos para las
mismas 202 imágenes del conjunto Validation.

El orden de las muestras y las etiquetas es idéntico al utilizado para
ResNet50V2.

El conjunto Test continúa completamente aislado.


### 9.1. Rutas del checkpoint y archivo de resultados

Definición de rutas para el checkpoint de Fine-Tuning de EfficientNetV2-M y verificación de su existencia en disco.


In [15]:
EFFICIENTNET_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "glaucoma"
    / "efficientnetv2m"
    / "efficientnetv2m_finetune_best.keras"
)
EFFICIENTNET_SCORES_PATH = (
    COMPARISON_RESULTS_DIR
    / "efficientnetv2m_validation_scores.csv"
)

print(
    "Checkpoint EfficientNet:",
    EFFICIENTNET_MODEL_PATH
)
print(
    "Existe:",
    EFFICIENTNET_MODEL_PATH.exists()
)
print(
    "Salida:",
    EFFICIENTNET_SCORES_PATH
)


Checkpoint EfficientNet: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/models/glaucoma/efficientnetv2m/efficientnetv2m_finetune_best.keras
Existe: True
Salida: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/efficientnetv2m_validation_scores.csv


### 9.2. Carga del modelo EfficientNetV2-M

Carga del mejor checkpoint de Fine-Tuning de EfficientNetV2-M con `compile=False` e inspección del número total de parámetros.


In [16]:
efficientnet_model = tf.keras.models.load_model(
    EFFICIENTNET_MODEL_PATH,
    compile=False
)

print(
    "Modelo cargado:",
    efficientnet_model.name
)
print(
    "Parámetros:",
    f"{efficientnet_model.count_params():,}"
)


Modelo cargado: EfficientNetV2M_Chaksu
Parámetros: 53,151,669


### 9.3. Generación de scores continuos sobre Validation

Inferencia sobre `val_ds_efficientnet` para obtener las estimaciones sigmoid de riesgo de glaucoma en las 202 imágenes de validación y comprobación de integridad numérica.


In [17]:
efficientnet_scores = efficientnet_model.predict(
    val_ds_efficientnet,
    verbose=1
)
efficientnet_scores = (
    efficientnet_scores
    .reshape(-1)
)

print(
    "Scores generados:",
    len(efficientnet_scores)
)
print(
    "Etiquetas:",
    len(y_true)
)
print(
    "Score mínimo:",
    float(efficientnet_scores.min())
)
print(
    "Score máximo:",
    float(efficientnet_scores.max())
)
print(
    "NaN:",
    int(
        np.isnan(
            efficientnet_scores
        ).sum()
    )
)


E0000 00:00:1790717465.945342  121571 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717467.498050  121571 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717468.234805  121571 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717468.983630  121571 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717470.534489  121571 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


25/26 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step

E0000 00:00:1790717490.839686  121569 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717491.853855  121569 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717492.703675  121569 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717493.522941  121569 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1790717494.712864  121569 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


26/26 ━━━━━━━━━━━━━━━━━━━━ 40s 894ms/step
Scores generados: 202
Etiquetas: 202
Score mínimo: 0.002849184675142169
Score máximo: 0.9686630964279175
NaN: 0


### 9.4. Evaluación discriminativa directa (ROC-AUC y Average Precision)

Cálculo independiente de ROC-AUC y Average Precision para EfficientNetV2-M sobre el conjunto de validación utilizando `scikit-learn`.


In [18]:
efficientnet_roc_auc_exact = (
    roc_auc_score(
        y_true,
        efficientnet_scores
    )
)
efficientnet_average_precision = (
    average_precision_score(
        y_true,
        efficientnet_scores
    )
)

print(
    "ROC-AUC directo:",
    f"{efficientnet_roc_auc_exact:.6f}"
)
print(
    "Average Precision:",
    f"{efficientnet_average_precision:.6f}"
)


ROC-AUC directo: 0.847196
Average Precision: 0.447088


### 9.5. Persistencia de predicciones individuales

Guardado de `efficientnetv2m_validation_scores.csv` con columnas `image_name`, `label` y `efficientnetv2m_score`.


In [19]:
efficientnet_validation_results = (
    pd.DataFrame(
        {
            "image_name": val_df[
                "image_name"
            ].values,
            "label": y_true,
            "efficientnetv2m_score": (
                efficientnet_scores
            )
        }
    )
)
efficientnet_validation_results.to_csv(
    EFFICIENTNET_SCORES_PATH,
    index=False
)

print(
    "Archivo guardado en:",
    EFFICIENTNET_SCORES_PATH
)
display(
    efficientnet_validation_results.head()
)


Archivo guardado en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/efficientnetv2m_validation_scores.csv


,image_name,label,efficientnetv2m_score
0,IMG_2544.JPG,0,0.020879
1,IMG_3380.JPG,0,0.019508
2,IMG_3506.JPG,0,0.028615
3,IMG_3625.JPG,0,0.024303
4,Image188.JPG,1,0.564197


### 9.6. Liberación de memoria GPU

Eliminación de la referencia al modelo EfficientNetV2-M, limpieza de backend y recolección de basura para liberar la VRAM.


In [20]:
del efficientnet_model
tf.keras.backend.clear_session()
gc.collect()

print(
    "EfficientNetV2-M liberado de memoria."
)


EfficientNetV2-M liberado de memoria.


## 10. Consolidación de predicciones de Validation

Se integran los scores generados por ResNet50V2 y EfficientNetV2-M
sobre las mismas 202 imágenes del conjunto Validation.

La unión se realiza mediante `image_name` y se verifica que las
etiquetas asociadas a cada imagen sean idénticas en ambos resultados.

Esta tabla constituye la base para la comparación estadística directa
entre ambas arquitecturas.


### 10.1. Carga, fusión y verificación de predicciones

Unión de ambos resultados por `image_name`, comprobación booleana de consistencia en etiquetas (`label`), selección de columnas consolidadas y persistencia en `glaucoma_validation_model_scores.csv`.


In [21]:
resnet_results = pd.read_csv(
    RESNET_SCORES_PATH
)
efficientnet_results = pd.read_csv(
    EFFICIENTNET_SCORES_PATH
)

comparison_scores = resnet_results.merge(
    efficientnet_results,
    on="image_name",
    how="inner",
    suffixes=(
        "_resnet",
        "_efficientnet"
    )
)

print(
    "Filas ResNet:",
    len(resnet_results)
)
print(
    "Filas EfficientNet:",
    len(efficientnet_results)
)
print(
    "Filas después de unir:",
    len(comparison_scores)
)

labels_coinciden = np.array_equal(
    comparison_scores[
        "label_resnet"
    ],
    comparison_scores[
        "label_efficientnet"
    ]
)
print(
    "Etiquetas coinciden:",
    labels_coinciden
)

if not labels_coinciden:
    raise ValueError(
        "Las etiquetas no coinciden entre modelos."
    )

comparison_scores["label"] = (
    comparison_scores[
        "label_resnet"
    ]
)
comparison_scores = comparison_scores[
    [
        "image_name",
        "label",
        "resnet50v2_score",
        "efficientnetv2m_score"
    ]
].copy()

COMBINED_SCORES_PATH = (
    COMPARISON_RESULTS_DIR
    / "glaucoma_validation_model_scores.csv"
)
comparison_scores.to_csv(
    COMBINED_SCORES_PATH,
    index=False
)

print(
    "Comparación guardada en:",
    COMBINED_SCORES_PATH
)
display(
    comparison_scores.head()
)


Filas ResNet: 202
Filas EfficientNet: 202
Filas después de unir: 202
Etiquetas coinciden: True
Comparación guardada en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/glaucoma_validation_model_scores.csv


,image_name,label,resnet50v2_score,efficientnetv2m_score
0,IMG_2544.JPG,0,0.006071,0.020879
1,IMG_3380.JPG,0,0.000201,0.019508
2,IMG_3506.JPG,0,0.021984,0.028615
3,IMG_3625.JPG,0,0.014642,0.024303
4,Image188.JPG,1,0.438085,0.564197


### 10.2. Comparación directa de métricas globales en Validation

Cálculo simultáneo y pareado de ROC-AUC y Average Precision sobre las mismas 202 observaciones de validación, computando la diferencia neta de rendimiento (ResNet50V2 − EfficientNetV2-M).


In [22]:
y_val = comparison_scores[
    "label"
].to_numpy()
scores_resnet = comparison_scores[
    "resnet50v2_score"
].to_numpy()
scores_efficientnet = comparison_scores[
    "efficientnetv2m_score"
].to_numpy()

exact_metrics = pd.DataFrame(
    {
        "ResNet50V2": {
            "ROC-AUC": roc_auc_score(
                y_val,
                scores_resnet
            ),
            "Average Precision": average_precision_score(
                y_val,
                scores_resnet
            )
        },
        "EfficientNetV2-M": {
            "ROC-AUC": roc_auc_score(
                y_val,
                scores_efficientnet
            ),
            "Average Precision": average_precision_score(
                y_val,
                scores_efficientnet
            )
        }
    }
)

exact_metrics[
    "Diferencia ResNet - EfficientNet"
] = (
    exact_metrics["ResNet50V2"]
    - exact_metrics["EfficientNetV2-M"]
)

display(
    exact_metrics.round(6)
)


,ResNet50V2,EfficientNetV2-M,Diferencia ResNet - EfficientNet
ROC-AUC,0.887407,0.847196,0.040212
Average Precision,0.544996,0.447088,0.097908


## 11. Intervalos de confianza mediante bootstrap pareado

Se aplica bootstrap pareado sobre Validation para estimar la
incertidumbre de ROC-AUC y Average Precision.

En cada repetición se remuestrean los mismos índices para ambas
arquitecturas. De esta manera, la comparación conserva el carácter
pareado del experimento, dado que los dos modelos fueron evaluados
sobre exactamente las mismas imágenes.

Se utilizan 2000 repeticiones y un nivel de confianza del 95 %.


### 11.1. Procedimiento de remuestreo bootstrap pareado (2000 repeticiones)

Generación de 2000 réplicas con reemplazo evaluando a ambos modelos sobre los mismos índices remuestreados para preservar la correlación de errores.


In [23]:
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(
    BOOTSTRAP_SEED
)
n_samples = len(
    comparison_scores
)
bootstrap_rows = []

for _ in range(N_BOOTSTRAP):
    indices = rng.integers(
        0,
        n_samples,
        size=n_samples
    )
    y_boot = y_val[
        indices
    ]
    # ROC-AUC requiere presencia
    # de ambas clases
    if np.unique(y_boot).size < 2:
        continue
    resnet_boot = scores_resnet[
        indices
    ]
    efficientnet_boot = (
        scores_efficientnet[
            indices
        ]
    )

    roc_resnet = roc_auc_score(
        y_boot,
        resnet_boot
    )
    roc_efficientnet = roc_auc_score(
        y_boot,
        efficientnet_boot
    )
    ap_resnet = average_precision_score(
        y_boot,
        resnet_boot
    )
    ap_efficientnet = average_precision_score(
        y_boot,
        efficientnet_boot
    )

    bootstrap_rows.append(
        {
            "roc_resnet": roc_resnet,
            "roc_efficientnet": roc_efficientnet,
            "roc_difference": (
                roc_resnet - roc_efficientnet
            ),
            "ap_resnet": ap_resnet,
            "ap_efficientnet": ap_efficientnet,
            "ap_difference": (
                ap_resnet - ap_efficientnet
            )
        }
    )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

print(
    "Repeticiones válidas:",
    len(bootstrap_df)
)


Repeticiones válidas: 2000


### 11.2. Intervalos de confianza al 95 %

Cálculo de los percentiles 2.5 % y 97.5 % para cada modelo y para la diferencia pareada, visualizando la estimación puntual y sus márgenes de incertidumbre.


In [24]:
def intervalo_95(
    values
):
    return np.percentile(
        values,
        [2.5, 97.5]
    )

roc_resnet_ci = intervalo_95(
    bootstrap_df[
        "roc_resnet"
    ]
)
roc_efficientnet_ci = intervalo_95(
    bootstrap_df[
        "roc_efficientnet"
    ]
)
roc_difference_ci = intervalo_95(
    bootstrap_df[
        "roc_difference"
    ]
)
ap_resnet_ci = intervalo_95(
    bootstrap_df[
        "ap_resnet"
    ]
)
ap_efficientnet_ci = intervalo_95(
    bootstrap_df[
        "ap_efficientnet"
    ]
)
ap_difference_ci = intervalo_95(
    bootstrap_df[
        "ap_difference"
    ]
)

ci_summary = pd.DataFrame(
    {
        "Estimación puntual": [
            exact_metrics.loc["ROC-AUC", "ResNet50V2"],
            exact_metrics.loc["ROC-AUC", "EfficientNetV2-M"],
            exact_metrics.loc["ROC-AUC", "Diferencia ResNet - EfficientNet"],
            exact_metrics.loc["Average Precision", "ResNet50V2"],
            exact_metrics.loc["Average Precision", "EfficientNetV2-M"],
            exact_metrics.loc["Average Precision", "Diferencia ResNet - EfficientNet"]
        ],
        "IC 95% Inferior": [
            roc_resnet_ci[0],
            roc_efficientnet_ci[0],
            roc_difference_ci[0],
            ap_resnet_ci[0],
            ap_efficientnet_ci[0],
            ap_difference_ci[0]
        ],
        "IC 95% Superior": [
            roc_resnet_ci[1],
            roc_efficientnet_ci[1],
            roc_difference_ci[1],
            ap_resnet_ci[1],
            ap_efficientnet_ci[1],
            ap_difference_ci[1]
        ]
    },
    index=[
        "ROC-AUC ResNet50V2",
        "ROC-AUC EfficientNetV2-M",
        "Diferencia ROC-AUC",
        "AP ResNet50V2",
        "AP EfficientNetV2-M",
        "Diferencia AP"
    ]
)

display(
    ci_summary.round(6)
)

print(
    f"ROC-AUC ResNet50V2: {exact_metrics.loc['ROC-AUC', 'ResNet50V2']:.4f} [95% CI: {roc_resnet_ci[0]:.4f}, {roc_resnet_ci[1]:.4f}]"
)
print(
    f"ROC-AUC EfficientNetV2-M: {exact_metrics.loc['ROC-AUC', 'EfficientNetV2-M']:.4f} [95% CI: {roc_efficientnet_ci[0]:.4f}, {roc_efficientnet_ci[1]:.4f}]"
)
print(
    f"Diferencia ROC-AUC: {exact_metrics.loc['ROC-AUC', 'Diferencia ResNet - EfficientNet']:.4f} [95% CI: {roc_difference_ci[0]:.4f}, {roc_difference_ci[1]:.4f}]"
)
print()
print(
    f"AP ResNet50V2: {exact_metrics.loc['Average Precision', 'ResNet50V2']:.4f} [95% CI: {ap_resnet_ci[0]:.4f}, {ap_resnet_ci[1]:.4f}]"
)
print(
    f"AP EfficientNetV2-M: {exact_metrics.loc['Average Precision', 'EfficientNetV2-M']:.4f} [95% CI: {ap_efficientnet_ci[0]:.4f}, {ap_efficientnet_ci[1]:.4f}]"
)
print(
    f"Diferencia AP: {exact_metrics.loc['Average Precision', 'Diferencia ResNet - EfficientNet']:.4f} [95% CI: {ap_difference_ci[0]:.4f}, {ap_difference_ci[1]:.4f}]"
)


,Estimación puntual,IC 95% Inferior,IC 95% Superior
ROC-AUC ResNet50V2,0.887407,0.816221,0.943164
ROC-AUC EfficientNetV2-M,0.847196,0.764142,0.917842
Diferencia ROC-AUC,0.040212,-0.020324,0.113731
AP ResNet50V2,0.544996,0.374096,0.759185
AP EfficientNetV2-M,0.447088,0.288117,0.658483
Diferencia AP,0.097908,-0.035513,0.245304


ROC-AUC ResNet50V2: 0.8874 [95% CI: 0.8162, 0.9432]
ROC-AUC EfficientNetV2-M: 0.8472 [95% CI: 0.7641, 0.9178]
Diferencia ROC-AUC: 0.0402 [95% CI: -0.0203, 0.1137]

AP ResNet50V2: 0.5450 [95% CI: 0.3741, 0.7592]
AP EfficientNetV2-M: 0.4471 [95% CI: 0.2881, 0.6585]
Diferencia AP: 0.0979 [95% CI: -0.0355, 0.2453]


## 12. Selección de la arquitectura para el modelo de riesgo de Glaucoma

La comparación experimental directa sobre las 202 imágenes del conjunto
Validation mostró mayores estimaciones puntuales para ResNet50V2 tanto
en ROC-AUC como en Average Precision.

ResNet50V2 obtuvo un ROC-AUC de 0.8874, mientras que EfficientNetV2-M
obtuvo 0.8472. Asimismo, ResNet50V2 alcanzó una Average Precision de
0.5450 frente a 0.4471 de EfficientNetV2-M.

El análisis mediante bootstrap pareado con 2000 repeticiones mostró que
los intervalos de confianza del 95 % de las diferencias entre
arquitecturas incluyeron el valor cero. Por tanto, estos resultados no
permiten afirmar una superioridad estadísticamente concluyente de
ResNet50V2 sobre EfficientNetV2-M en la partición Validation.

No obstante, conforme al criterio de selección previamente establecido,
ResNet50V2 es seleccionada como arquitectura del modelo especializado en
evaluación de riesgo de Glaucoma, debido a que presentó la mayor
estimación puntual del criterio principal (ROC-AUC), la mayor estimación
del criterio secundario (Average Precision) y una menor complejidad
paramétrica.

A partir de este punto, EfficientNetV2-M deja de intervenir en la
selección del umbral y en la evaluación final. El conjunto Test permanece
sin utilizar.


In [25]:
SELECTED_MODEL = "ResNet50V2"

selected_model_summary = pd.DataFrame(
    {
        "Valor": {
            "Arquitectura seleccionada": (
                SELECTED_MODEL
            ),
            "ROC-AUC Validation": (
                roc_auc_score(
                    y_val,
                    scores_resnet
                )
            ),
            "Average Precision Validation": (
                average_precision_score(
                    y_val,
                    scores_resnet
                )
            ),
            "Parámetros totales": (
                23_566_849
            )
        }
    }
)

display(
    selected_model_summary
)


,Valor
Arquitectura seleccionada,ResNet50V2
ROC-AUC Validation,0.887407
Average Precision Validation,0.544996
Parámetros totales,23566849


## 13. Selección del umbral operativo en Validation

Una vez seleccionada ResNet50V2 como arquitectura del modelo de riesgo
de Glaucoma, se determina su umbral operativo utilizando exclusivamente
el conjunto Validation.

El valor 0.5 empleado durante el entrenamiento corresponde únicamente a
un umbral provisional y no se considera automáticamente como el punto
de operación definitivo.

En ausencia de un objetivo clínico de sensibilidad previamente definido,
el umbral se selecciona maximizando la Balanced Accuracy, con el objetivo
de equilibrar sensibilidad y especificidad frente al desbalance entre
LOW_RISK y HIGH_RISK.

El conjunto Test permanece completamente aislado durante este proceso.


### 13.1. Barrido de umbrales sobre la curva ROC

Cálculo de la curva ROC (`roc_curve`), especificidad, Balanced Accuracy y Youden J para cada punto de corte posible sobre las 202 predicciones continuas de ResNet50V2.


In [26]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(
    y_val,
    scores_resnet
)

specificity = 1 - fpr
balanced_accuracy = (
    tpr + specificity
) / 2
youden_j = (
    tpr + specificity - 1
)

threshold_analysis = pd.DataFrame(
    {
        "threshold": thresholds,
        "sensitivity": tpr,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "youden_j": youden_j
    }
)

threshold_analysis = (
    threshold_analysis
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
    .reset_index(
        drop=True
    )
)

best_threshold_idx = (
    threshold_analysis[
        "balanced_accuracy"
    ].idxmax()
)
best_threshold_row = (
    threshold_analysis.loc[
        best_threshold_idx
    ]
)

display(
    best_threshold_row
)


threshold            0.100155
sensitivity          0.962963
specificity          0.702857
balanced_accuracy    0.832910
youden_j             0.665820
Name: 29, dtype: float64

### 13.2. Determinación del umbral operativo óptimo

Extracción y reporte de los parámetros de corte correspondientes al máximo de Balanced Accuracy / Youden J sobre el conjunto Validation.


In [27]:
SELECTED_THRESHOLD = float(
    best_threshold_row[
        "threshold"
    ]
)

print(
    "Umbral seleccionado:",
    f"{SELECTED_THRESHOLD:.6f}"
)
print(
    "Sensibilidad:",
    f"{best_threshold_row['sensitivity']:.6f}"
)
print(
    "Especificidad:",
    f"{best_threshold_row['specificity']:.6f}"
)
print(
    "Balanced Accuracy:",
    f"{best_threshold_row['balanced_accuracy']:.6f}"
)
print(
    "Youden J:",
    f"{best_threshold_row['youden_j']:.6f}"
)


Umbral seleccionado: 0.100155
Sensibilidad: 0.962963
Especificidad: 0.702857
Balanced Accuracy: 0.832910
Youden J: 0.665820


### 13.3. Métricas de clasificación binaria bajo el umbral congelado

Binarización de los scores continuos de ResNet50V2 aplicando `SELECTED_THRESHOLD`, cálculo de la matriz de confusión (TN, FP, FN, TP) y cómputo de métricas clínicas y operativas.


In [28]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    f1_score,
    balanced_accuracy_score
)

y_pred_selected = (
    scores_resnet
    >= SELECTED_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    y_pred_selected
).ravel()

sensitivity_selected = (
    tp / (tp + fn)
)
specificity_selected = (
    tn / (tn + fp)
)
precision_selected = precision_score(
    y_val,
    y_pred_selected,
    zero_division=0
)
f1_selected = f1_score(
    y_val,
    y_pred_selected,
    zero_division=0
)
balanced_accuracy_selected = (
    balanced_accuracy_score(
        y_val,
        y_pred_selected
    )
)

validation_threshold_metrics = pd.DataFrame(
    {
        "Métrica": [
            "Umbral seleccionado",
            "Sensibilidad (Recall)",
            "Especificidad",
            "Balanced Accuracy",
            "Precision",
            "F1-Score",
            "Verdaderos Negativos (TN)",
            "Falsos Positivos (FP)",
            "Falsos Negativos (FN)",
            "Verdaderos Positivos (TP)"
        ],
        "Valor": [
            SELECTED_THRESHOLD,
            sensitivity_selected,
            specificity_selected,
            balanced_accuracy_selected,
            precision_selected,
            f1_selected,
            int(tn),
            int(fp),
            int(fn),
            int(tp)
        ]
    }
)

display(
    validation_threshold_metrics.round(6)
)


,Métrica,Valor
0,Umbral seleccionado,0.100155
1,Sensibilidad (Recall),0.962963
2,Especificidad,0.702857
3,Balanced Accuracy,0.832910
4,Precision,0.333333
5,F1-Score,0.495238
6,Verdaderos Negativos (TN),123.000000
7,Falsos Positivos (FP),52.000000
8,Falsos Negativos (FN),1.000000
9,Verdaderos Positivos (TP),26.000000


## 14. Congelamiento del modelo y umbral operativo

Finalizada la selección utilizando exclusivamente Validation, se congela
la configuración que será utilizada para la evaluación final sobre Test.

Arquitectura seleccionada: ResNet50V2.

Umbral operativo seleccionado mediante máxima Balanced Accuracy en
Validation: 0.100155 aproximadamente.

A partir de este punto no se realizarán modificaciones adicionales a la
arquitectura, pesos del modelo ni umbral de decisión a partir de los
resultados obtenidos sobre Test.


In [29]:
import json

FINAL_MODEL_NAME = SELECTED_MODEL
FINAL_THRESHOLD = SELECTED_THRESHOLD
FINAL_CONFIG_PATH = (
    COMPARISON_RESULTS_DIR
    / "glaucoma_final_model_config.json"
)

final_config = {
    "model_name": FINAL_MODEL_NAME,
    "model_path": str(RESNET_MODEL_PATH),
    "decision_threshold": float(FINAL_THRESHOLD),
    "threshold_selection_criterion": (
        "max_balanced_accuracy_validation"
    ),
    "validation_samples": int(len(y_val)),
    "validation_roc_auc": float(
        roc_auc_score(
            y_val,
            scores_resnet
        )
    ),
    "validation_average_precision": float(
        average_precision_score(
            y_val,
            scores_resnet
        )
    ),
    "validation_sensitivity": float(
        sensitivity_selected
    ),
    "validation_specificity": float(
        specificity_selected
    ),
    "validation_balanced_accuracy": float(
        balanced_accuracy_selected
    ),
    "validation_precision": float(
        precision_selected
    ),
    "validation_f1": float(
        f1_selected
    ),
    "validation_tn": int(tn),
    "validation_fp": int(fp),
    "validation_fn": int(fn),
    "validation_tp": int(tp)
}

with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Modelo congelado:",
    FINAL_MODEL_NAME
)
print(
    "Threshold congelado:",
    f"{FINAL_THRESHOLD:.6f}"
)
print(
    "Configuración guardada en:",
    FINAL_CONFIG_PATH
)


Modelo congelado: ResNet50V2
Threshold congelado: 0.100155
Configuración guardada en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/glaucoma_final_model_config.json


## 15. Evaluación final sobre Test

Con la arquitectura ResNet50V2 y el umbral operativo previamente
seleccionados utilizando exclusivamente Validation, se realiza la
evaluación final sobre el conjunto Test oficial de Chákṣu.

El conjunto Test contiene 336 imágenes y no fue utilizado durante el
entrenamiento, Fine-Tuning, selección de arquitectura ni selección del
umbral.

El modelo, sus pesos y el umbral permanecen congelados durante toda
esta evaluación.


### 15.1. Recuperar la configuración congelada

Carga del archivo `glaucoma_final_model_config.json` para recuperar el nombre del modelo, la ruta del checkpoint y el umbral operativo óptimo sin reajustes manuales.


In [30]:
import json

with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    final_config_loaded = json.load(f)

FINAL_MODEL_NAME = (
    final_config_loaded.get(
        "model",
        final_config_loaded.get("model_name")
    )
)
FINAL_MODEL_PATH = Path(
    final_config_loaded.get(
        "checkpoint",
        final_config_loaded.get("model_path")
    )
)
FINAL_THRESHOLD = float(
    final_config_loaded.get(
        "threshold",
        final_config_loaded.get("decision_threshold")
    )
)

print(
    "Modelo final:",
    FINAL_MODEL_NAME
)
print(
    "Checkpoint:",
    FINAL_MODEL_PATH
)
print(
    "Existe checkpoint:",
    FINAL_MODEL_PATH.exists()
)
print(
    "Threshold congelado:",
    f"{FINAL_THRESHOLD:.6f}"
)


Modelo final: ResNet50V2
Checkpoint: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/models/glaucoma/resnet50v2/resnet50v2_finetune_best.keras
Existe checkpoint: True
Threshold congelado: 0.100155


### 15.2. Preparación del conjunto Test y construcción del dataset

Extracción de las 336 muestras oficiales del conjunto Test y construcción del dataset con `tf.data` aplicando el preprocesamiento geométrico y escalamiento de ResNet50V2 (`[-1, 1]`) sin Data Augmentation ni desordenamiento (`shuffle=False`).


In [31]:
test_df = splits_df[
    splits_df["experimental_split"]
    == "test"
].copy()
test_df = test_df.reset_index(
    drop=True
)

print(
    "Imágenes Test:",
    len(test_df)
)
print(
    "\nDistribución:"
)
print(
    test_df["label"]
    .value_counts()
    .sort_index()
)

test_ds_resnet = (
    crear_dataset_base(
        test_df,
        shuffle=False
    )
    .map(
        preprocess_resnet50v2,
        num_parallel_calls=NUM_PARALLEL_CALLS
    )
    .batch(BATCH_SIZE)
    .prefetch(PREFETCH_BUFFER)
)

for images_test, labels_test in (
    test_ds_resnet.take(1)
):
    print(
        "\nShape:",
        images_test.shape
    )
    print(
        "Min/Max:",
        float(
            tf.reduce_min(
                images_test
            )
        ),
        float(
            tf.reduce_max(
                images_test
            )
        )
    )
    print(
        "Labels:",
        labels_test.numpy()
    )


Imágenes Test: 336

Distribución:
label
0    285
1     51
Name: count, dtype: int64

Shape: (8, 384, 384, 3)
Min/Max: -1.0 1.0
Labels: [0. 0. 0. 0. 0. 0. 0. 0.]


## 16. Inferencia del modelo seleccionado sobre Test

Carga del checkpoint congelado de ResNet50V2 y generación de scores continuos sobre las 336 imágenes de Test, garantizando la integridad numérica de las salidas.


In [32]:
final_model = tf.keras.models.load_model(
    FINAL_MODEL_PATH,
    compile=False
)

print(
    "Modelo cargado:",
    final_model.name
)
print(
    "Parámetros:",
    f"{final_model.count_params():,}"
)

test_scores = final_model.predict(
    test_ds_resnet,
    verbose=1
)
test_scores = (
    test_scores
    .reshape(-1)
)

y_test = (
    test_df["label"]
    .astype(int)
    .to_numpy()
)

print(
    "Scores generados:",
    len(test_scores)
)
print(
    "Etiquetas:",
    len(y_test)
)
print(
    "Score mínimo:",
    float(test_scores.min())
)
print(
    "Score máximo:",
    float(test_scores.max())
)
print(
    "NaN:",
    int(np.isnan(test_scores).sum())
)


Modelo cargado: ResNet50V2_Chaksu
Parámetros: 23,566,849
42/42 ━━━━━━━━━━━━━━━━━━━━ 22s 452ms/step
Scores generados: 336
Etiquetas: 336
Score mínimo: 0.00014583462325390428
Score máximo: 0.9425690174102783
NaN: 0


## 17. Métricas independientes del umbral (ROC-AUC y Average Precision)

Evaluación de la capacidad discriminativa global de ResNet50V2 sobre el conjunto Test mediante ROC-AUC y Average Precision a partir de los scores continuos sin binarizar.


In [33]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    confusion_matrix,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

test_roc_auc = roc_auc_score(
    y_test,
    test_scores
)
test_average_precision = (
    average_precision_score(
        y_test,
        test_scores
    )
)

print(
    "ROC-AUC Test:",
    f"{test_roc_auc:.6f}"
)
print(
    "Average Precision Test:",
    f"{test_average_precision:.6f}"
)


ROC-AUC Test: 0.819539
Average Precision Test: 0.422710


## 18. Evaluación de clasificación binaria bajo el umbral congelado

Binarización de los scores continuos aplicando rigurosamente el umbral `FINAL_THRESHOLD` obtenido en Validation, cálculo de la matriz de confusión y determinación de las métricas de generalización final.


In [34]:
y_test_pred = (
    test_scores
    >= FINAL_THRESHOLD
).astype(int)

tn_test, fp_test, fn_test, tp_test = (
    confusion_matrix(
        y_test,
        y_test_pred
    ).ravel()
)

test_sensitivity = (
    tp_test
    / (tp_test + fn_test)
)
test_specificity = (
    tn_test
    / (tn_test + fp_test)
)
test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)
test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)
test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)
test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        y_test_pred
    )
)

test_metrics = pd.DataFrame(
    {
        "Métrica": [
            "Threshold congelado",
            "ROC-AUC",
            "Average Precision",
            "Accuracy",
            "Sensibilidad (Recall)",
            "Especificidad",
            "Balanced Accuracy",
            "Precision",
            "F1-Score",
            "Verdaderos Negativos (TN)",
            "Falsos Positivos (FP)",
            "Falsos Negativos (FN)",
            "Verdaderos Positivos (TP)"
        ],
        "Valor": [
            FINAL_THRESHOLD,
            test_roc_auc,
            test_average_precision,
            test_accuracy,
            test_sensitivity,
            test_specificity,
            test_balanced_accuracy,
            test_precision,
            test_f1,
            int(tn_test),
            int(fp_test),
            int(fn_test),
            int(tp_test)
        ]
    }
)

display(
    test_metrics.round(6)
)


,Métrica,Valor
0,Threshold congelado,0.100155
1,ROC-AUC,0.819539
2,Average Precision,0.422710
3,Accuracy,0.741071
4,Sensibilidad (Recall),0.745098
5,Especificidad,0.740351
6,Balanced Accuracy,0.742724
7,Precision,0.339286
8,F1-Score,0.466258
9,Verdaderos Negativos (TN),211.000000


## 19. Persistencia de resultados finales sobre Test

Almacenamiento de las predicciones individuales a nivel de imagen (`resnet50v2_test_scores.csv`) y de las métricas agregadas de rendimiento (`resnet50v2_test_metrics.csv`) en el directorio de resultados.


In [35]:
TEST_SCORES_PATH = (
    COMPARISON_RESULTS_DIR
    / "resnet50v2_test_scores.csv"
)
test_results = pd.DataFrame(
    {
        "image_name": (
            test_df["image_name"].values
        ),
        "label": (
            y_test
        ),
        "score": (
            test_scores
        ),
        "predicted_label": (
            y_test_pred
        )
    }
)
test_results.to_csv(
    TEST_SCORES_PATH,
    index=False
)

print(
    "Scores Test guardados en:",
    TEST_SCORES_PATH
)

TEST_METRICS_PATH = (
    COMPARISON_RESULTS_DIR
    / "resnet50v2_test_metrics.csv"
)
test_metrics.to_csv(
    TEST_METRICS_PATH,
    index=False
)

print(
    "Métricas Test guardadas en:",
    TEST_METRICS_PATH
)

display(
    test_results.head()
)


Scores Test guardados en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/resnet50v2_test_scores.csv
Métricas Test guardadas en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/resnet50v2_test_metrics.csv


,image_name,label,score,predicted_label
0,P10_Image2.JPG,0,0.146862,1
1,P11_Image1.JPG,0,0.033138,0
2,P11_Image2.JPG,0,0.179750,1
3,P12_Image1.JPG,0,0.006688,0
4,P12_Image2.JPG,0,0.413803,1


## 20. Intervalos de confianza de la evaluación final

Se cuantifica la incertidumbre de las métricas finales mediante bootstrap
sobre el conjunto Test.

Esta etapa tiene únicamente finalidad descriptiva y de estimación de
incertidumbre. Los resultados obtenidos no se utilizan para modificar
el modelo, sus pesos ni el umbral operativo previamente congelado.

Se emplean 2000 remuestreos bootstrap y un intervalo de confianza del
95 %.


### 20.1. Procedimiento bootstrap sobre Test (2000 repeticiones)

Remuestreo con reemplazo para generar 2000 réplicas del conjunto Test, calculando en cada iteración métricas continuas (ROC-AUC, AP) y discretas bajo el umbral congelado.


In [36]:
N_BOOTSTRAP_TEST = 2000
BOOTSTRAP_TEST_SEED = 42
rng_test = np.random.default_rng(
    BOOTSTRAP_TEST_SEED
)
n_test = len(
    test_df
)
bootstrap_test_rows = []

for _ in range(N_BOOTSTRAP_TEST):
    indices = rng_test.integers(
        0,
        n_test,
        size=n_test
    )
    y_boot = y_test[
        indices
    ]
    if np.unique(y_boot).size < 2:
        continue
    scores_boot = test_scores[
        indices
    ]
    pred_boot = (
        scores_boot
        >= FINAL_THRESHOLD
    ).astype(int)

    tn_b, fp_b, fn_b, tp_b = (
        confusion_matrix(
            y_boot,
            pred_boot
        ).ravel()
    )
    sensitivity_b = (
        tp_b / (tp_b + fn_b)
    )
    specificity_b = (
        tn_b / (tn_b + fp_b)
    )

    bootstrap_test_rows.append(
        {
            "roc_auc": (
                roc_auc_score(
                    y_boot,
                    scores_boot
                )
            ),
            "average_precision": (
                average_precision_score(
                    y_boot,
                    scores_boot
                )
            ),
            "sensitivity": (
                sensitivity_b
            ),
            "specificity": (
                specificity_b
            ),
            "balanced_accuracy": (
                balanced_accuracy_score(
                    y_boot,
                    pred_boot
                )
            ),
            "precision": (
                precision_score(
                    y_boot,
                    pred_boot,
                    zero_division=0
                )
            ),
            "f1": (
                f1_score(
                    y_boot,
                    pred_boot,
                    zero_division=0
                )
            )
        }
    )

bootstrap_test_df = pd.DataFrame(
    bootstrap_test_rows
)

print(
    "Repeticiones válidas:",
    len(bootstrap_test_df)
)


Repeticiones válidas: 2000


### 20.2. Estimación y reporte de intervalos de confianza al 95 %

Cálculo de los percentiles 2.5 % y 97.5 % para cada métrica final, estructurando la tabla comparativa entre estimación puntual e intervalo empírico.


In [37]:
def resumen_ic95(
    nombre,
    valor_puntual,
    valores_boot
):
    ic_inf, ic_sup = np.percentile(
        valores_boot,
        [2.5, 97.5]
    )
    return {
        "Métrica": nombre,
        "Estimación puntual": valor_puntual,
        "IC 95% Inferior": ic_inf,
        "IC 95% Superior": ic_sup
    }

test_ci_summary = pd.DataFrame(
    [
        resumen_ic95(
            "ROC-AUC",
            test_roc_auc,
            bootstrap_test_df[
                "roc_auc"
            ]
        ),
        resumen_ic95(
            "Average Precision",
            test_average_precision,
            bootstrap_test_df[
                "average_precision"
            ]
        ),
        resumen_ic95(
            "Sensibilidad",
            test_sensitivity,
            bootstrap_test_df[
                "sensitivity"
            ]
        ),
        resumen_ic95(
            "Especificidad",
            test_specificity,
            bootstrap_test_df[
                "specificity"
            ]
        ),
        resumen_ic95(
            "Balanced Accuracy",
            test_balanced_accuracy,
            bootstrap_test_df[
                "balanced_accuracy"
            ]
        ),
        resumen_ic95(
            "Precision",
            test_precision,
            bootstrap_test_df[
                "precision"
            ]
        ),
        resumen_ic95(
            "F1-Score",
            test_f1,
            bootstrap_test_df[
                "f1"
            ]
        )
    ]
)

display(
    test_ci_summary.round(6)
)


,Métrica,Estimación puntual,IC 95% Inferior,IC 95% Superior
0,ROC-AUC,0.819539,0.759363,0.872265
1,Average Precision,0.422710,0.307742,0.565031
2,Sensibilidad,0.745098,0.622631,0.862767
3,Especificidad,0.740351,0.688161,0.792860
4,Balanced Accuracy,0.742724,0.678389,0.806367
5,Precision,0.339286,0.253927,0.428571
6,F1-Score,0.466258,0.367347,0.557382


### 20.3. Persistencia de intervalos de confianza

Guardado del archivo `resnet50v2_test_confidence_intervals.csv` con los intervalos de confianza finales de la evaluación.


In [38]:
TEST_CI_PATH = (
    COMPARISON_RESULTS_DIR
    / "resnet50v2_test_confidence_intervals.csv"
)
test_ci_summary.to_csv(
    TEST_CI_PATH,
    index=False
)

print(
    "Intervalos guardados en:",
    TEST_CI_PATH
)


Intervalos guardados en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/resnet50v2_test_confidence_intervals.csv


## 21. Conclusión del experimento de Glaucoma

Se evaluaron experimentalmente ResNet50V2 y EfficientNetV2-M para la
clasificación binaria de riesgo de Glaucoma utilizando las mismas
particiones de Chákṣu, el mismo preprocesamiento geométrico y una
estrategia común de Transfer Learning y Fine-Tuning.

La comparación directa sobre Validation mostró mayores estimaciones
puntuales para ResNet50V2 tanto en ROC-AUC como en Average Precision.
ResNet50V2 obtuvo un ROC-AUC de 0.8874 y una Average Precision de
0.5450, frente a 0.8472 y 0.4471, respectivamente, para
EfficientNetV2-M.

El análisis mediante bootstrap pareado no mostró una diferencia
estadísticamente concluyente entre ambas arquitecturas, debido a que
los intervalos de confianza del 95 % de las diferencias incluyeron el
valor cero. No obstante, de acuerdo con el criterio de selección
previamente establecido, ResNet50V2 fue seleccionada por presentar
las mayores estimaciones puntuales en las métricas principales y una
menor complejidad paramétrica.

Posteriormente, utilizando exclusivamente Validation, se seleccionó
un umbral operativo de 0.100155 mediante la maximización de Balanced
Accuracy. Este umbral, junto con la arquitectura y los pesos del
modelo, fue congelado antes de utilizar el conjunto Test.

La evaluación final independiente sobre las 336 imágenes de Test
obtuvo un ROC-AUC de 0.8195 (IC 95 %: 0.7594–0.8723), una Average
Precision de 0.4227 (IC 95 %: 0.3077–0.5650), una sensibilidad de
0.7451 (IC 95 %: 0.6226–0.8628), una especificidad de 0.7404
(IC 95 %: 0.6882–0.7929) y una Balanced Accuracy de 0.7427
(IC 95 %: 0.6784–0.8064).

Con el umbral congelado, el modelo clasificó correctamente 38 de los
51 casos HIGH_RISK y 211 de los 285 casos LOW_RISK, registrándose
13 falsos negativos y 74 falsos positivos.

Estos resultados corresponden a una evaluación experimental del
modelo sobre el dataset Chákṣu y no constituyen por sí solos evidencia
de desempeño clínico en población de Lima Metropolitana. La validación
posterior con datos locales y participación de especialistas será
necesaria para determinar su capacidad de generalización en el entorno
objetivo de la solución.


In [39]:
final_glaucoma_summary = pd.DataFrame(
    {
        "Configuración / Métrica": [
            "Arquitectura final",
            "Dataset",
            "Resolución",
            "Threshold",
            "ROC-AUC Test",
            "Average Precision Test",
            "Sensibilidad Test",
            "Especificidad Test",
            "Balanced Accuracy Test",
            "Precision Test",
            "F1-Score Test",
            "TN",
            "FP",
            "FN",
            "TP"
        ],
        "Resultado": [
            "ResNet50V2",
            "Chákṣu",
            "384 × 384",
            f"{FINAL_THRESHOLD:.6f}",
            f"{test_roc_auc:.6f}",
            f"{test_average_precision:.6f}",
            f"{test_sensitivity:.6f}",
            f"{test_specificity:.6f}",
            f"{test_balanced_accuracy:.6f}",
            f"{test_precision:.6f}",
            f"{test_f1:.6f}",
            int(tn_test),
            int(fp_test),
            int(fn_test),
            int(tp_test)
        ]
    }
)

display(
    final_glaucoma_summary
)

FINAL_SUMMARY_PATH = (
    COMPARISON_RESULTS_DIR
    / "glaucoma_final_experiment_summary.csv"
)
final_glaucoma_summary.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)

print(
    "Resumen final guardado en:",
    FINAL_SUMMARY_PATH
)


,Configuración / Métrica,Resultado
0,Arquitectura final,ResNet50V2
1,Dataset,Chákṣu
2,Resolución,384 × 384
3,Threshold,0.100155
4,ROC-AUC Test,0.819539
5,Average Precision Test,0.422710
6,Sensibilidad Test,0.745098
7,Especificidad Test,0.740351
8,Balanced Accuracy Test,0.742724
9,Precision Test,0.339286


Resumen final guardado en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/glaucoma/model_comparison/glaucoma_final_experiment_summary.csv
